# ComplianceGuard — GRPO Training Notebook

**Stack:** Unsloth + TRL GRPO | **Model:** Qwen2.5-1.5B-Instruct 4-bit QLoRA | **GPU:** T4 (free Colab)

**Meta × HuggingFace OpenEnv Hackathon — Theme 3.1 Professional Tasks**

> The model outputs ALL actions in one completion → executed through a real `ComplianceGuardEnv` instance → reward varies 0.05–1.0 → genuine learning signal.  
> No oracle, no shortcuts. If it misses PII, the reward reflects it.

**Health check (step 5):** `reward_std` must be > 0.05. If it is 0.000, training is broken.

**Results after 200 steps on T4:** reward trend +66% (first-10 avg 0.097 → last-10 avg 0.161), peak 0.298 at step 181.

## Cell 1 — Install dependencies

In [ ]:
!pip install unsloth --quiet
!pip install 'trl>=0.15.0' datasets accelerate peft --quiet

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram = f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else ''
print(f'GPU: {gpu}  {vram}')
if not torch.cuda.is_available():
    raise RuntimeError('Switch to GPU: Runtime > Change runtime type > T4 GPU')

## Cell 2 — ComplianceGuard environment (inline, no server)

3-phase FSM: **SCAN → CLASSIFY → REDACT**. Reward = harmonic mean of scan_f1, classify_accuracy, redact_completeness.
Phase gates are enforced — calling a wrong-phase tool returns an error and a penalty.

In [ ]:
import json, random

_FIRST = ['Alice','Bob','Carol','David','Emma','Frank','Grace','Henry','Iris','Jack']
_LAST  = ['Johnson','Martinez','White','Brown','Davis','Wilson','Anderson','Taylor']
_DOMS  = ['gmail.com','yahoo.com','company.com','corp.io','work.net']

def _name():
    f, l = random.choice(_FIRST), random.choice(_LAST)
    return f'{f} {l}', f, l

def _email(f, l): return f'{f.lower()}.{l.lower()}@{random.choice(_DOMS)}'

def _phone():
    a,m,e = random.randint(200,999), random.randint(100,999), random.randint(1000,9999)
    return random.choice([f'({a}) {m}-{e}', f'{a}.{m}.{e}', f'+1-{a}-{m}-{e}'])

def _obf_email(email):
    u,d = email.split('@'); p = d.split('.')
    return f'{u} [at] {p[0]} [dot] {p[1]}'

def _obf_phone(phone):
    return phone.replace('-',' dash ').replace('.',' dot ').replace('(','').replace(')','').strip()

def gen_l1():
    nm,f,l = _name(); email,phone = _email(f,l), _phone()
    c = (f'2026-01-01 INFO user {nm} logged in from 10.0.0.5\n'
         f'2026-01-01 INFO support contact: {email}\n'
         f'2026-01-02 WARN call us at {phone} for help\n')
    return {'server_logs.txt': c}, [nm, email, phone]

def gen_l2():
    pp=[_name() for _ in range(3)]
    names=[p[0] for p in pp]; emails=[_email(p[1],p[2]) for p in pp]; phones=[_phone() for _ in range(3)]
    logs  = (f'2026-01-01 ERROR user {names[0]} failed login\n'
             f'2026-01-01 INFO support: {emails[0]}\n'
             f'2026-01-02 INFO {names[1]} called {phones[0]}\n')
    report= (f'Incident report\nContact: {names[2]}\n'
             f'Email: {emails[1]}\nPhone: {phones[1]}\n'
             f'Alt email: {emails[2]}\nAlt phone: {phones[2]}\n')
    return {'server_logs.txt': logs, 'incident_report.txt': report}, names+emails+phones

def gen_l3():
    pp=[_name() for _ in range(3)]
    names=[p[0] for p in pp]; re2=[_email(p[1],p[2]) for p in pp]; rp=[_phone() for _ in range(3)]
    oe=[_obf_email(e) for e in re2]; op=[_obf_phone(p) for p in rp]
    logs  = f'2026-01-01 INFO contact {names[0]} via {oe[0]}\n2026-01-02 WARN reach {names[1]} at {op[0]}\n'
    report= f'User: {names[2]}\nReach at: {oe[1]}\nPhone: {op[1]}\n'
    notes = f'Backup contacts: {oe[2]} / {op[2]}\nServer IP: 192.168.1.1\n'
    return {'server_logs.txt': logs, 'incident_report.txt': report, 'notes.txt': notes}, names+oe+op

def gen_l4():
    files, pii = gen_l3()
    rh = ['test@example.com','noreply@system.local','admin@localhost','000.000.0000']
    fn = list(files.keys())
    files[fn[0]] += f'System alert sent to {rh[0]}\nAuto-reply from {rh[1]}\n'
    files['system_log.txt'] = f'Automated notification: {rh[2]}\nTest phone: {rh[3]}\n'
    return files, pii

GENERATORS = {1: gen_l1, 2: gen_l2, 3: gen_l3, 4: gen_l4}
PHASE_TOOLS = {
    'SCAN':     ['list_files','read_file','flag_candidate','advance_phase'],
    'CLASSIFY': ['list_candidates','classify_candidate','advance_phase'],
    'REDACT':   ['redact_span','submit'],
}


class ComplianceGuardEnv:
    def __init__(self): self.reset()

    def reset(self, seed=None, level=1, **kw):
        if seed is not None: random.seed(seed)
        self.level = level
        self.virtual_fs, self.pii_list = GENERATORS.get(level, gen_l1)()
        self.candidates = {}; self._cid = 0; self.phase = 'SCAN'
        self.done = False; self.reward = 0.0; self.cumulative = 0.0
        self._pending_step_reward = 0.0
        return f'Reset L{level}. Files: {list(self.virtual_fs.keys())}. Use list_files -> read_file -> flag_candidate -> advance_phase.'

    def step(self, action_json):
        self._pending_step_reward = 0.0
        try: parsed = json.loads(action_json); tool = parsed.get('tool','')
        except: return self._obs(-0.05,'Error: send valid JSON with tool field.')
        fn = getattr(self, f'_t_{tool}', None)
        if fn is None: return self._obs(-0.05, f'Unknown tool {tool!r}. Allowed: {PHASE_TOOLS[self.phase]}')
        try: result = fn(parsed)
        except ValueError as e: return self._obs(-0.05, str(e))
        step_reward = self._pending_step_reward
        if self.done: return self._obs(self.reward, result, done=True)
        return self._obs(step_reward, result)

    def _req(self, phase, tool):
        if self.phase != phase: raise ValueError(f'{tool!r} only in {phase}. Currently {self.phase}.')

    def _t_list_files(self, p):
        self._req('SCAN','list_files'); return f'Files: {list(self.virtual_fs.keys())}'

    def _t_read_file(self, p):
        self._req('SCAN','read_file'); fp=p.get('file_path','')
        if fp not in self.virtual_fs: raise ValueError(f'{fp!r} not found.')
        return f'=== {fp} ===\n{self.virtual_fs[fp]}'

    def _t_flag_candidate(self, p):
        self._req('SCAN','flag_candidate'); text=p.get('text','').strip()
        if not text: raise ValueError('text required.')
        existing=[c for c,v in self.candidates.items() if v['text'].strip()==text]
        if existing: return f'Already flagged as {existing[0]}.'
        is_real_pii=any(text in pii or pii in text for pii in self.pii_list)
        self._pending_step_reward = 0.04 if is_real_pii else -0.02
        cid=f'c{self._cid}'; self._cid+=1
        self.candidates[cid]={'text':text,'file_path':p.get('file_path',''),'pii_type':p.get('pii_type','OTHER'),'confirmed':None,'redacted':False}
        hint=' [real PII]' if is_real_pii else ' [not in PII list]'
        return f'Flagged {cid}: {p.get("pii_type","OTHER")} | {text!r}{hint}'

    def _t_advance_phase(self, p):
        if self.phase=='SCAN':
            if not self.candidates: raise ValueError('No candidates. Flag PII first.')
            self.phase='CLASSIFY'; return f'-> CLASSIFY. {len(self.candidates)} candidates.'
        elif self.phase=='CLASSIFY':
            unc=[c for c,v in self.candidates.items() if v['confirmed'] is None]
            if unc: raise ValueError(f'Classify all first. Unclassified: {unc}')
            self.phase='REDACT'; conf=[c for c,v in self.candidates.items() if v['confirmed']]
            return f'-> REDACT. {len(conf)} confirmed PII.'
        else: raise ValueError('Already in REDACT. Use redact_span + submit.')

    def _t_list_candidates(self, p):
        self._req('CLASSIFY','list_candidates')
        lines=[f'  {c}: [{"CONFIRMED" if v["confirmed"] else ("REJECTED" if v["confirmed"]==False else "PENDING")}] {v["pii_type"]} | {v["text"]!r}' for c,v in self.candidates.items()]
        return 'Candidates:\n'+('\n'.join(lines) or 'none')

    def _t_classify_candidate(self, p):
        self._req('CLASSIFY','classify_candidate'); cid=p.get('candidate_id','')
        if cid not in self.candidates: raise ValueError(f'Unknown {cid}.')
        c=p.get('confirmed')
        if c is None: raise ValueError('confirmed required.')
        confirmed=bool(c)
        text=self.candidates[cid]['text']
        is_real_pii=any(text in pii or pii in text for pii in self.pii_list)
        correct=confirmed==is_real_pii
        self._pending_step_reward=0.02 if correct else -0.03
        self.candidates[cid]['confirmed']=confirmed
        return f'{cid} {"CONFIRMED" if confirmed else "REJECTED"}: {text!r}'

    def _t_redact_span(self, p):
        self._req('REDACT','redact_span'); cid=p.get('candidate_id','')
        if cid not in self.candidates: raise ValueError(f'Unknown {cid}.')
        c=self.candidates[cid]
        if not c['confirmed']: raise ValueError(f'{cid} not confirmed.')
        if c['redacted']: return f'{cid} already redacted.'
        text,fp = c['text'],c['file_path']
        if fp and fp in self.virtual_fs and text in self.virtual_fs[fp]:
            self.virtual_fs[fp]=self.virtual_fs[fp].replace(text,'[REDACTED]')
        else:
            for fname in self.virtual_fs:
                if text in self.virtual_fs[fname]: self.virtual_fs[fname]=self.virtual_fs[fname].replace(text,'[REDACTED]')
        c['redacted']=True
        self._pending_step_reward=0.03
        return f'Redacted {cid}: {text!r} -> [REDACTED]'

    def _t_submit(self, p):
        self._req('REDACT','submit'); self.reward,metrics=self._compute_reward(); self.done=True
        return (f'Done. reward={self.reward:.4f} | '
                f'scan_f1={metrics.get("scan_f1",0):.3f} '
                f'classify_acc={metrics.get("classify_accuracy",0):.3f} '
                f'redact_complete={metrics.get("redact_completeness",0):.3f} '
                f'harmonic={metrics.get("harmonic_mean",0):.3f}')

    def _compute_reward(self):
        total = len(self.pii_list)
        if total == 0: return 0.5, {}
        flagged_texts = {v['text'] for v in self.candidates.values()}
        tp_scan = sum(1 for pii in self.pii_list
                      if pii in flagged_texts or any(pii in ft or ft in pii for ft in flagged_texts))
        fp_scan = sum(1 for ft in flagged_texts
                      if not any(ft in pii or pii in ft for pii in self.pii_list))
        scan_recall    = tp_scan / total
        scan_precision = tp_scan / max(1, tp_scan + fp_scan)
        scan_f1 = 2.0 * scan_precision * scan_recall / max(1e-9, scan_precision + scan_recall)
        classified = [v for v in self.candidates.values() if v['confirmed'] is not None]
        if classified:
            correct = sum(1 for v in classified
                          if bool(v['confirmed']) == any(v['text'] in p or p in v['text'] for p in self.pii_list))
            classify_acc = correct / len(classified)
        else:
            classify_acc = 0.0
        all_content = '\n'.join(self.virtual_fs.values())
        still_present = sum(1 for p in self.pii_list if p in all_content)
        redact_complete = 1.0 - (still_present / total)
        components = [scan_f1, classify_acc, redact_complete]
        if all(c > 1e-9 for c in components):
            harmonic = len(components) / sum(1.0 / c for c in components)
        else:
            harmonic = 0.0
        if harmonic >= 0.99:   reward = 0.999
        elif harmonic > 0:     reward = 0.05 + 0.949 * harmonic
        else:                  reward = 0.05
        return max(0.001, min(0.999, reward)), {
            'scan_f1': round(scan_f1, 4),
            'classify_accuracy': round(classify_acc, 4),
            'redact_completeness': round(redact_complete, 4),
            'harmonic_mean': round(harmonic, 4),
        }

    def _obs(self, reward, result, done=False):
        self.cumulative += reward
        return {'phase': self.phase, 'result': result, 'reward': reward,
                'cumulative': round(self.cumulative, 4), 'done': done, 'pii_count': len(self.pii_list)}

    def close(self): pass


# Sanity check
env = ComplianceGuardEnv()
env.reset(seed=42, level=1)
obs = env.step(json.dumps({'tool':'list_files'}))
assert 'server_logs.txt' in obs['result'], 'Env broken!'
print('ComplianceGuardEnv OK:', obs['result'])

## Cell 3 — Reward function (real episodes, no oracle)

Model outputs ALL actions in one completion. We execute them through a fresh env and return real episode reward.  
**`extract_json_actions`** handles single-line, multi-line, and markdown-fenced JSON.  
**Phase-aware partial credit** gives gradient signal even when episode doesn’t reach `submit`.  
The unit test at the end **must print `Reward function verified`** before you run training.

In [ ]:
import re

_debug_printed = False

def extract_json_actions(text: str) -> list:
    actions = []
    # Strip markdown code fences
    text = re.sub(r'```(?:json)?\s*', '', text)
    text = re.sub(r'```', '', text)

    # Pass 1: single-line JSON objects
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('{') and line.endswith('}'):
            try:
                json.loads(line)
                actions.append(line)
            except Exception:
                pass

    if actions:
        return actions

    # Pass 2: balanced-brace scan (handles pretty-printed multi-line JSON)
    depth, start = 0, -1
    for i, ch in enumerate(text):
        if ch == '{':
            if depth == 0: start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start != -1:
                try:
                    obj = json.loads(text[start:i+1])
                    actions.append(json.dumps(obj, separators=(',', ':')))
                except Exception:
                    pass
                start = -1
    return actions


def run_episode_from_actions(action_strings: list, level: int, seed: int) -> float:
    env = ComplianceGuardEnv()
    env.reset(seed=seed, level=level)

    if not action_strings:
        return 0.05

    for action_str in action_strings[:40]:
        try:
            obs = env.step(action_str)
        except Exception:
            continue
        if obs.get('done'):
            return float(env.reward)

    # Phase-aware partial credit
    # Harmonic mean kills gradient when any component=0; give explicit per-phase signal instead
    partial_r, metrics = env._compute_reward()
    scan_f1 = metrics.get('scan_f1', 0.0)

    if env.phase == 'REDACT':
        return max(0.07, 0.05 + 0.6 * partial_r)   # near-complete episode
    elif env.phase == 'CLASSIFY':
        return max(0.06, 0.05 + 0.3 * scan_f1)      # good scan, didn't finish
    elif scan_f1 > 0:
        return max(0.055, 0.05 + 0.1 * scan_f1)     # found some PII in scan
    else:
        return 0.05


def compliance_reward(completions, prompts=None, **kwargs):
    global _debug_printed
    levels = list(kwargs.get('level', []))
    seeds  = list(kwargs.get('seed',  []))
    rewards = []

    for i, comp in enumerate(completions):
        if isinstance(comp, list):
            text = comp[0].get('content', '') if comp else ''
        elif isinstance(comp, dict):
            text = comp.get('content', '')
        else:
            text = str(comp)

        # One-time debug print so you can see exactly what the model generates
        if not _debug_printed and i == 0:
            print(f'\n=== DEBUG: comp type={type(comp).__name__} ===')
            print(f'text (first 600):\n{text[:600]}')
            acts = extract_json_actions(text)
            print(f'extracted {len(acts)} actions: {acts[:4]}')
            print('=== END DEBUG ===\n')
            _debug_printed = True

        level = int(levels[i]) if i < len(levels) else random.randint(1, 3)
        seed  = int(seeds[i])  if i < len(seeds)  else i * 7 + 42

        try:
            actions = extract_json_actions(text)
            r = run_episode_from_actions(actions, level=level, seed=seed)
        except Exception:
            r = 0.05

        rewards.append(float(r))

    return rewards


# ── Unit test ─────────────────────────────────────────────────────────────────────────
random.seed(42)
_e = ComplianceGuardEnv(); _e.reset(seed=42, level=1)
_pii = _e.pii_list[:]; _files = list(_e.virtual_fs.keys())
_acts_perfect = (
    [json.dumps({'tool':'list_files'}),
     json.dumps({'tool':'read_file','file_path':_files[0]})]
    + [json.dumps({'tool':'flag_candidate','text':p,'file_path':_files[0],'pii_type':'NAME'}) for p in _pii]
    + [json.dumps({'tool':'advance_phase'})]
    + [json.dumps({'tool':'classify_candidate','candidate_id':f'c{j}','confirmed':True}) for j in range(len(_pii))]
    + [json.dumps({'tool':'advance_phase'})]
    + [json.dumps({'tool':'redact_span','candidate_id':f'c{j}'}) for j in range(len(_pii))]
    + [json.dumps({'tool':'submit'})]
)

r_perfect = run_episode_from_actions(_acts_perfect, level=1, seed=42)
r_empty   = run_episode_from_actions([], level=1, seed=42)
print(f'Perfect: {r_perfect:.4f}  Empty: {r_empty:.4f}')
assert r_perfect > 0.9, f'BROKEN: {r_perfect}'
assert r_empty == 0.05
print('Reward function verified -- safe to run training')

## Cell 4 — System prompt and dataset

Prompt is **list-of-dicts** (chat format) with full file contents in the user message.  
This is the critical fix vs. naive string prompts — the model can’t flag PII it hasn’t seen.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a PII compliance agent. You will be given file contents containing PII.
Identify and redact all personally identifiable information.

PII: person names, email addresses, phone numbers, SSNs.
NOT PII: dates (2026-01-01), IPs (192.168.1.1), system emails (test@example.com, noreply@*, admin@localhost), log levels.

3-phase workflow: SCAN -> CLASSIFY -> REDACT

SCAN:
  {"tool": "list_files"}
  {"tool": "read_file", "file_path": "<name>"}
  {"tool": "flag_candidate", "text": "<exact text from file>", "file_path": "<name>", "pii_type": "<NAME|EMAIL|PHONE|SSN>"}
  {"tool": "advance_phase"}
CLASSIFY:
  {"tool": "list_candidates"}
  {"tool": "classify_candidate", "candidate_id": "<id>", "confirmed": true}
  {"tool": "advance_phase"}
REDACT:
  {"tool": "redact_span", "candidate_id": "<id>"}
  {"tool": "submit"}

Output ALL actions ONE JSON per line. No prose. Use EXACT text from the files."""


def make_prompt(level: int, seed: int) -> list:
    env = ComplianceGuardEnv()
    env.reset(seed=seed, level=level)

    files_text = ''
    for fname, content in env.virtual_fs.items():
        files_text += f'\n--- {fname} ---\n{content}'

    user_msg = f'Files to audit:{files_text}\nFind and redact ALL PII. Output one JSON action per line.'
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg}
    ]


def build_dataset(n_per_level: int = 50) -> Dataset:
    rows = []
    for level in [1, 1, 1, 2, 2, 3]:   # weighted toward easy levels for early learning signal
        for seed in range(n_per_level):
            s = seed + level * 1000
            rows.append({'prompt': make_prompt(level, s), 'level': level, 'seed': s})
    random.shuffle(rows)
    return Dataset.from_list(rows)


dataset = build_dataset(n_per_level=50)
print(f'Dataset: {len(dataset)} rows')
print('User message (first 500 chars):')
print(dataset[0]['prompt'][1]['content'][:500])

## Cell 5 — Load model (T4-safe)

`max_seq_length=2048` is required — the prompt is ~800 tokens. Using 512 causes a `TorchRuntimeError`.

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOConfig, GRPOTrainer

# Must patch BEFORE using GRPOTrainer
PatchFastRL('GRPO', FastLanguageModel)

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=2048,      # must be >=2048 -- prompt is ~800 tokens, 512 causes TorchRuntimeError
    load_in_4bit=True,
    fast_inference=False,     # required on T4 -- no vLLM
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Model: Qwen2.5-1.5B-Instruct 4-bit QLoRA')
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Cell 6 — GRPO Training

**After step 5, check `reward_std` — it must be > 0.05.**  
If reward=0.999 and reward_std=0.000 from step 1, training is broken (oracle hack — re-check Cell 3).

In [ ]:
config = GRPOConfig(
    output_dir='checkpoints/grpo',
    max_steps=200,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    use_vllm=False,               # required on T4
    max_completion_length=384,
    logging_steps=1,
    save_steps=50,
    report_to='none',
)

# NOTE: use args= not config= (Unsloth patched GRPOTrainer renamed the param)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=compliance_reward,
)

print('Starting GRPO training...')
print('HEALTH CHECK: reward_std should be > 0.05 after step 5.')
print('If reward=0.999 and reward_std=0.000 from step 1 -> training is broken (oracle hack).')
trainer.train()
print('Training complete!')

# Save reward log
import csv
log = trainer.state.log_history

def _get_reward(e):
    return e.get('reward', e.get('train/reward', e.get('rewards')))

reward_rows = [{'step': e['step'], 'reward': _get_reward(e)} for e in log if _get_reward(e) is not None]
if reward_rows:
    with open('reward_log.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['step','reward'])
        w.writeheader(); w.writerows(reward_rows)
    vals = [r['reward'] for r in reward_rows]
    print(f'reward_log.csv saved ({len(vals)} rows)')
    print(f'  step-1 reward:  {vals[0]:.4f}')
    print(f'  first-10 avg:   {sum(vals[:10])/10:.4f}')
    print(f'  last-10 avg:    {sum(vals[-10:])/len(vals[-10:]):.4f}')
    print(f'  peak reward:    {max(vals):.4f} (step {reward_rows[vals.index(max(vals))]["step"]})')

## Cell 7 — Save model

In [ ]:
model.save_pretrained('checkpoints/grpo/final-adapter')
tokenizer.save_pretrained('checkpoints/grpo/final-adapter')
print('Adapter saved: checkpoints/grpo/final-adapter')

model.save_pretrained_merged('checkpoints/grpo/final-merged', tokenizer, save_method='merged_16bit')
print('Merged 16-bit saved: checkpoints/grpo/final-merged')

## Cell 8 — Plot reward curve

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, csv

with open('reward_log.csv') as f:
    rows = list(csv.DictReader(f))
steps   = [int(r['step'])    for r in rows]
rewards = [float(r['reward']) for r in rows]
print(f'Loaded {len(rewards)} entries')

def smooth(v, w=10): return np.convolve(v, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('ComplianceGuard GRPO Training', fontsize=14, fontweight='bold')

axes[0].plot(steps, rewards, alpha=0.25, color='steelblue', lw=0.8, label='raw')
if len(rewards) >= 10:
    sw = smooth(rewards)
    axes[0].plot(steps[:len(sw)], sw, color='steelblue', lw=2, label='smoothed (w=10)')
axes[0].axhline(0.70, color='green', ls='--', lw=1.5, label='success threshold (0.70)')
axes[0].set(xlabel='Step', ylabel='Reward', title='Episode Reward')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(-0.05, 1.1)

cm = np.cumsum(rewards) / (np.arange(len(rewards)) + 1)
axes[1].plot(steps, cm, color='darkorange', lw=2, label='cumulative mean')
axes[1].axhline(0.70, color='green', ls='--', lw=1.5, label='success threshold')
axes[1].set(xlabel='Step', ylabel='Cumulative mean reward', title='Cumulative Mean')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
print('Saved reward_curve.png')

## Cell 9 — Download results

In [ ]:
from google.colab import files
files.download('reward_log.csv')
files.download('reward_curve.png')
print('Downloaded reward_log.csv and reward_curve.png')

## Troubleshooting

| Error | Fix |
|-------|-----|
| `reward=0.999, reward_std=0.000` from step 1 | Oracle hacking — check Cell 3 unit test passes |
| `TorchRuntimeError` / `index out of range` | `max_seq_length` too small — use 2048 (Cell 5) |
| `ImportError: vLLM` | `fast_inference=False` in Cell 5 |
| `TypeError: got unexpected kwarg 'config'` | Use `args=config` not `config=config` in Cell 6 |
| CUDA OOM | Lower `per_device_train_batch_size` to 1 and `max_completion_length` to 256 |
| `reward_std=0.000` but reward ≠ 0.999 | Model only outputs non-JSON text — check Cell 4 prompt includes file contents |